# `melt_gamma_tuning_ssa`

> Generated from `config/ais_0.1_SSA.py` by `config/make_step_notebooks.py` on 2026-10-29. Edit the driver and regenerate; changes made here are overwritten.

Apply the validated SSA ocean-melt calibration (gamma_0=300, refit deltaT).

**Reads** (`load_model` calls in this step):
- `f'{model_dir}/AIS3_SSA_relaxed.nc'`

**Writes** (`save_model` calls in this step):
- `out_path = f"{model_dir}/AIS3_melt_final_ssa{os.environ.get('AIS3_MELT_TAG', '')}.nc"`

**Run as a batch job** (from `config/notebooks/`; resources copied from `config/launch_ais_0.1_SSA.pbs`):

```bash
qsub -q normal -l ncpus=8,mem=190GB,walltime=25:00:00 -v NB=ais_0.1_SSA/02_melt_gamma_tuning_ssa.ipynb launch_notebook.pbs
```

Add run options to the `-v` list, e.g. `NB=...,AIS3_MELT_TAG=<value>`. The executed copy is written next to this notebook as `executed/<name>.<jobid>.ipynb`. Interactive use needs a Jupyter session with at least those resources.

**Run options**: set them in the next cell (interactive) or with `qsub -v` (batch).

Status, results and known failures: [`docs/run_guide.md`](../../../docs/run_guide.md) and the worklogs in `docs/`.

### What this step does
Builds `AIS3_melt_final_ssa.nc`, the input to the SSA historical and projection steps, from
the validated SSA melt calibration. There is no solve. It loads `AIS3_SSA_relaxed.nc` as the
model shell and sets ISMIP6 quadratic-local melt:
- IMBIE2 basins and Zhou thermal forcing
- `gamma_0=300` and the SSA track's 16 refit `deltaT` values, hard-coded in the step

### Before you run
- The step refuses to overwrite an existing output. Archive it (`mv -n`) or set
  `AIS3_MELT_TAG`.
- To redo the calibration rather than apply it, run the standalone chain (run guide §6):
  1. `melt_gamma_sweep_ssa.py`
  2. `melt_deltaT_sensitivity_test_ssa.py`
  3. `ssa_melt_deltaT_basin_refit.py`
  4. `ssa_melt_deltaT_refit_recalibrate.py`

  Then copy the new values into this step.

### What to check afterwards
- 30 thermal-forcing depth layers, and `gamma_0=300` in the saved file.

### Known pitfalls
- `AIS3_melt_gamma_tuning_ssa.nc` (`gamma_0=5537.7`) is the superseded coarse pick. Nothing
  should read it.
- At `gamma_0=300`, melt responds to ocean warming only about 4% as strongly as the ISMIP7
  ocean models (worklog §6.1). Projections use `gamma_0=15000` (`AIS3_PROJ_MELT=high`) as the
  high-melt member.

_Hand-written notes: `config/notebooks/notes/melt_gamma_tuning_ssa.md` (edit there, then regenerate)._

In [ ]:
import os

# Uncomment to set an option for an interactive run. Values passed with
# `qsub -v` already arrive in the environment. Defaults shown are the code's own.
# os.environ['AIS3_MELT_TAG'] = ...   # code default: ''

## Setup

The preamble of `ais_0.1_SSA.py`: imports, helpers, settings and the data catalog.

#### Imports

In [ ]:
"""ACCESS-AIS3 pure-SSA production pipeline (Budd/Weertman friction law, p=q=1 -- the best
validated RMSE anywhere in this project, 60.4 m/yr, vs. the Schoof-based HO track's current
94-99 m/yr after several chunked-restart iterations). See docs/inversion_worklog.md for the
full history of how p=q=1 was found, and the "in your opinion" exchange in this session's
transcript for why a dedicated SSA production track is worth having alongside the HO one:
SSA is dramatically cheaper and already converged/stable, at the cost of no vertical-shear
physics (so it can't do thermal-coupled work the way the HO track can).

Deliberately self-contained -- does NOT import or exec ais_0.1.py, and does NOT touch its
shared `steps`/`save` globals. This project hit two real race-condition incidents this
session from toggling those globals while a job was mid-flight; a fully separate file with
its own local `steps`/`save` sidesteps that risk entirely, at the cost of some code
duplication (the handful of helper functions and Budd-specific step logic below are copied
from ais_0.1.py, not imported, so the two files can diverge safely and be run concurrently).

Friction source: models/AIS3_ssa_friction_inv_reg_lcurve/run_001_10_100_0.0001/ -- the real,
production-canonical location the RMSE=60.4 p=q=1 result was salvaged into (see
config/finalize_p1q1_reg_lcurve.py) from the ad-hoc script that originally produced it
(p1q1_reg_sweep2.py, no longer present as a file, only as raw execution output). This
finalization step recovers the ACTUAL computed result rather than re-deriving it from
scratch, which would otherwise require regenerating AIS3_param.nc with Budd config (it
currently has Schoof baked in, for the HO track) and re-running two expensive inversion
stages. Run finalize_p1q1_reg_lcurve.py once before this script's first stage.

Stage status (updated 2026-09-02):
  1. ssa_inverted_solve_budd -- FULLY IMPLEMENTED AND RUN (RMSE=65.67 m/yr full-continental,
     the best full-continental result in the project). AIS3_SSA_inverted.nc.
  2. ssa_relaxation_budd -- FULLY IMPLEMENTED AND RUN (mean |dH|=32.90m, max |dH|=1505.39m
     over the 20yr relaxation). AIS3_SSA_relaxed.nc.
  3. melt_gamma_tuning_ssa -- builds AIS3_melt_final_ssa.nc directly from the validated
     calibration (gamma_0=300 + per-basin refit deltaT_basin, hard-coded; 2026-09-29). The
     values come from the standalone refit chain (ssa_melt_deltaT_basin_refit.py /
     ssa_melt_deltaT_refit_recalibrate.py). The existing AIS3_melt_final_ssa.nc was built by
     finalize_ssa_melt_calibration.py. The step's old coarse gamma_0 sweep (pick 5537.7,
     AIS3_melt_gamma_tuning_ssa.nc) was superseded by that refit and removed.
  4. historical_dhdt_tuning_ssa -- RUN AND VALIDATED. Full 1995-2019 (24yr) transient against
     AIS3_melt_final_ssa.nc, checked against MIPKIT's dhdt_cpom: grounded RMSE=1.402 m/yr,
     independently reconfirmed via check_historical_ssa.py. AIS3_historical_1995_2019_SSA.nc.
  5. projection_ssp_ssa -- IMPLEMENTED (2026-09-29): anomaly forcing on the calibrated
     baseline (TF0 = Zhou, SMB0 = RACMO 1995-2014), CTRL/SSP/attribution/melt variants chosen
     per job via qsub -v, self-chaining 50-yr chunks to 2300. See launch_projection_ssa.pbs and
     docs/projection_experiments.md.

Both new stages use the plain `cluster` (48 cores/190GB/normal) config, not hugemem -- the
SSA mesh is ~15x smaller than the HO track's (no vertical layers), and both stages use the
same synchronous submit-and-wait pattern (`waitonlock` minutes + `load_only=False` in one
call) already validated throughout this pipeline, avoiding the "never actually submits" bug
found and fixed in ais_0.1.py's own ho_relaxation/historical_dhdt_tuning on their first real
run.
"""
import pyissm
import ccdtools as ccdtools
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
import os
import glob
from pathlib import Path
from scipy.spatial import Delaunay

os.chdir('/g/data/au88/jh7060/ACCESS-AIS3/')
os.environ['ISSM_DIR'] = '/g/data/vk83/apps/spack/1.1/release/linux-x86_64/issm-git.2026.05.18_2026.05.18-kgta35igm37z4qnqnul7rcmgx2inftqd'

model_dir = '/g/data/au88/jh7060/ACCESS-AIS3/models'
execution_dir = '/g/data/au88/jh7060/ACCESS-AIS3/execution_SSA'

save = True
diagnostics = True
plot = True

cluster = pyissm.model.classes.cluster.gadi()
cluster.codepath = os.environ['ISSM_DIR'] + '/bin'
cluster.executionpath = execution_dir
cluster.storage = 'gdata/au88+gdata/vk83+gdata/av17'
cluster.moduleuse = ['/g/data/vk83/modules/']
cluster.moduleload = ['access-issm_ad/2026.05.0']
cluster.np = 48
cluster.memory = 190
cluster.time = 60 * 48
cluster.login = 'jh7060'
cluster.project = 'au88'

FRICTION_RUN_DIR = f'{model_dir}/AIS3_ssa_friction_inv_reg_lcurve/run_001_10_100_0.0001'
FRICTION_RUN_NAME = 'run_001_10_100_0.0001'

all_steps = ['ssa_inverted_solve_budd', 'ssa_relaxation_budd', 'melt_gamma_tuning_ssa', 'historical_dhdt_tuning_ssa', 'projection_ssp_ssa']
steps = ['historical_dhdt_tuning_ssa']  # edit before each qsub -- see launch_ais_0.1_SSA.pbs
# Per-job override via `qsub -v AIS3_STEPS=a,b` (same as ais_0.1.py) -- lets a launcher pick its
# step without editing the shared `steps` above.
if os.environ.get('AIS3_STEPS'):
    steps = os.environ['AIS3_STEPS'].split(',')

#### `_interp_weights()` — Precompute a reusable linear-interpolation transform (Delaunay triangulation +

In [ ]:
def _interp_weights(src_x, src_y, tgt_x, tgt_y):
    """Precompute a reusable linear-interpolation transform (Delaunay triangulation +
    barycentric weights) ONCE for a fixed (source-grid, target-mesh) pair, so many
    different value arrays defined on that same source grid can each be interpolated in
    milliseconds afterward via _interp_apply.

    Needed for projection_ssp_ssa's time-varying ocean thermal forcing: 30 depths x ~81
    years means ~2430 interpolations of the SAME 761x761 source grid onto the SAME mesh --
    pyissm.data.interp.points_to_mesh (scipy.interpolate.griddata under the hood) rebuilds
    the full Delaunay triangulation AND redoes point-location for every single call, which
    is fine for this pipeline's existing handful-of-calls loops (RACMO's 24 annual SMB
    calls) but empirically verified to cost ~13-20s PER CALL even with triangulation reuse
    alone (evaluation/point-location dominates, not triangulation) -- at 2430 calls that is
    hours just for TF preprocessing, dwarfing the actual transient solve. Precomputing the
    per-target-point simplex + barycentric weights ONCE instead drops each subsequent
    interpolation to ~10ms (verified empirically), since applying already-known weights to
    a new value array is just a weighted sum, not a fresh geometric search.
    """
    src_xy = np.column_stack([np.asarray(src_x).ravel(), np.asarray(src_y).ravel()])
    tgt_xy = np.column_stack([np.asarray(tgt_x).ravel(), np.asarray(tgt_y).ravel()])
    tri = Delaunay(src_xy)
    simplex = tri.find_simplex(tgt_xy)
    vertices = np.take(tri.simplices, simplex, axis=0)
    temp = np.take(tri.transform, simplex, axis=0)
    delta = tgt_xy - temp[:, 2]
    bary = np.einsum('njk,nk->nj', temp[:, :2, :], delta)
    wts = np.hstack((bary, 1 - bary.sum(axis=1, keepdims=True)))
    return vertices, wts, simplex

#### `_interp_apply()` — Apply a transform from _interp_weights to a new value array defined on the same

In [ ]:
def _interp_apply(values, vtx, wts, simplex, fill_value=np.nan):
    """Apply a transform from _interp_weights to a new value array defined on the same
    source grid it was built from -- same interpolation this pipeline uses everywhere else
    (linear, NaN outside the source grid's convex hull), just without repeating the
    triangulation/point-location work each time."""
    values = np.asarray(values).ravel()
    out = np.einsum('nj,nj->n', values[vtx], wts)
    out[simplex == -1] = fill_value
    return out

#### `load_shelf_rheology_B()` — Identical to ais_0.1.py's own helper -- copied, not imported, deliberately (see

In [ ]:
def load_shelf_rheology_B():
    """Identical to ais_0.1.py's own helper -- copied, not imported, deliberately (see
    module docstring). The validated floating-shelf rheology result, independent of
    friction law entirely."""
    _cl = pyissm.model.classes.cluster.gadi()
    _cl.codepath = os.environ['ISSM_DIR'] + '/bin'
    _cl.executionpath = '/g/data/au88/jh7060/ACCESS-AIS3/execution_newB_rheology'
    _cl.login = 'jh7060'; _cl.project = 'au88'; _cl.storage = 'gdata/au88'
    mshelf = pyissm.model.io.load_model(f'{model_dir}/AIS3_param.nc')
    mshelf.mask.ice_levelset = pyissm.model.param.kill_icebergs(mshelf)
    sel = (mshelf.mask.ocean_levelset < 0) & (mshelf.mask.ice_levelset < 0)
    mshelf = mshelf.extract(sel)
    mshelf.cluster = _cl
    mshelf.settings.waitonlock = 0
    mshelf.inversion.iscontrol = 0
    mshelf.miscellaneous.name = 'run_001_1_10_1e-17'
    mr = pyissm.model.execute.solve(mshelf, 'Stressbalance', load_only=True, runtime_name=False, check_consistency=False)
    return np.asarray(mr.mesh.extractedvertices).ravel(), np.asarray(mr.results.StressbalanceSolution.MaterialsRheologyBbar).ravel()

## Step `melt_gamma_tuning_ssa`

In [ ]:
steps = ['melt_gamma_tuning_ssa']  # this notebook's step; overrides the driver default and AIS3_STEPS

**Section comment from the driver** (kept as written; it records history and may be out of date):

### Stage 3: Ocean basal melt -- apply the validated calibration, SSA mesh

Builds AIS3_melt_final_ssa.nc (historical_dhdt_tuning_ssa's input) from the validated ISMIP6 quadratic-local melt calibration: gamma_0 = 300 m/yr with per-basin refit deltaT_basin, official IMBIE2 basins, Zhou thermal-forcing climatology (worklog 4k, 4n). No solve: the melt parameterisation is evaluated inside the downstream transients.

The calibration SEARCH is not rerun here. It is a multi-round chain of standalone scripts (melt_gamma_sweep_ssa.py -> melt_deltaT_sensitivity_test_ssa.py -> ssa_melt_deltaT_basin_refit.py -> ssa_melt_deltaT_refit_recalibrate.py; state in models/deltaT_refit_state_ssa.json), and this step hard-codes its result. The J1-only gamma_0 sweep with fixed (published) deltaT_basin this step used to run was removed: its pick (5537.7) is superseded once deltaT_basin is refit per basin (J1 then ~0 for every candidate; J2 decides, worklog 4c).

Loads AIS3_SSA_relaxed.nc as the calibration did; its geometry is the inverted (unrelaxed) geometry, by design (worklog 4z correction). Thickness was already restored upstream in AIS3_SSA_inverted.nc (restore_floored_thickness_ssa.py, worklog 4k).

Refuses to overwrite an existing output: archive it first (mv -n) or set AIS3_MELT_TAG.

In [ ]:
print("-------------------------------------------------------------")
print(" OCEAN BASAL-MELT CALIBRATION, SSA (VALIDATED: gamma_0=300, REFIT deltaT)")
print("-------------------------------------------------------------")

from scipy.interpolate import RegularGridInterpolator, NearestNDInterpolator

ISMIP7_OCEAN = '/g/data/au88/ismip6/2300/forcings/ISMIP7/AIS/parameterisations/ocean'
ZHOU_TF_FILE = ('/g/data/au88/ismip6/2300/forcings/ISMIP7/AIS/obs/ocean/climatology/'
                 'zhou_annual_06_nov/tf/v3/tf_AIS_obs_ocean_climatology_zhou_annual_06_nov_v3_1972-2024.nc')
BASIN_FILE = f'{ISMIP7_OCEAN}/imbie2/basin_numbers_ismip8km_v2.nc'
TIME_SENTINEL = 1e9

# Validated values: candidate g1 of ssa_melt_deltaT_basin_refit.py (converged round 5, run
# AIS3_ssa_deltaT_refit_g1_r5), chosen by ssa_melt_deltaT_refit_recalibrate.py (job
# 179305728, J2 minimum). delta_t[k] belongs to IMBIE2 basin k+1. Close to, but not the
# same as, the HO track's refit values (different mesh).
MELT_GAMMA0 = 300.0
MELT_DELTA_T = np.array([
    1.8467609040725819, 2.7988123113211087, 2.2562599573273427, 4.599712312498511,
    8.762072283674417, 5.8733775833670565, 3.728768234826943, 0.8688693456218887,
    7.1878933110812575, 12.258715419441897, 5.2128687613028335, 4.89285368124807,
    2.3120433144628567, 3.4396433109318245, 1.2381000126240504, 2.2497361133862843])

out_path = f"{model_dir}/AIS3_melt_final_ssa{os.environ.get('AIS3_MELT_TAG', '')}.nc"
if save and os.path.exists(out_path):
    raise FileExistsError(f"{out_path} exists -- archive it first (mv -n) or set AIS3_MELT_TAG")

#### Loading relaxed SSA model

In [ ]:
print("-- Loading relaxed SSA model...")
md = pyissm.model.io.load_model(f'{model_dir}/AIS3_SSA_relaxed.nc')
md.results.TransientSolution = []  # 400-step relaxation history, unused downstream (worklog 4g)

#### Configuring ISMIP6 basal melt parameterisation (IMBIE2 basins)

In [ ]:
print("-- Configuring ISMIP6 basal melt parameterisation (IMBIE2 basins)...")
md.basalforcings = pyissm.model.classes.basalforcings.ismip6(md.basalforcings)
ds_basin = xr.open_dataset(BASIN_FILE)
basin_grid = ds_basin['basinNumber'].values
bxx, byy = np.meshgrid(ds_basin['x'].values, ds_basin['y'].values)
num_basins = int(np.nanmax(basin_grid)) + 1
assert num_basins == MELT_DELTA_T.size, f"{num_basins} basins vs {MELT_DELTA_T.size} delta_t values"
# basin_id is per element: nearest basin at each element centroid (2D mesh, no projection).
elx = np.asarray(md.mesh.elements).astype(int) - 1
mesh_x = np.asarray(md.mesh.x).ravel()
mesh_y = np.asarray(md.mesh.y).ravel()
ecx = mesh_x[elx[:, :3]].mean(axis=1)
ecy = mesh_y[elx[:, :3]].mean(axis=1)
basin_lookup = NearestNDInterpolator(np.column_stack([bxx.ravel(), byy.ravel()]), basin_grid.ravel())
md.basalforcings.basin_id = (basin_lookup(np.column_stack([ecx, ecy])) + 1).astype(float)
md.basalforcings.num_basins = num_basins
md.basalforcings.islocal = 1
md.basalforcings.gamma_0 = MELT_GAMMA0
md.basalforcings.delta_t = MELT_DELTA_T.copy()

#### Interpolating Zhou ocean thermal-forcing climatology onto the mesh

In [ ]:
print("-- Interpolating Zhou ocean thermal-forcing climatology onto the mesh...")
ds_tf = xr.open_dataset(ZHOU_TF_FILE)
tfx, tfy, tfz = ds_tf['x'].values, ds_tf['y'].values, ds_tf['z'].values
tf_full = ds_tf['tf'].values  # (z, y, x)
tf_list = []
for k in range(tfz.size):
    vals = RegularGridInterpolator((tfy, tfx), tf_full[k], method='linear',
                                    bounds_error=False, fill_value=np.nan)(np.column_stack([mesh_y, mesh_x]))
    tf_list.append(np.append(np.nan_to_num(vals, nan=0.0), TIME_SENTINEL).reshape(-1, 1))
md.basalforcings.tf = tf_list
md.basalforcings.tf_depths = tfz.copy()
print(f"   {tfz.size} depth layers ({tfz.min():.0f} to {tfz.max():.0f} m)")

# Forward-run settings the downstream transients inherit (they reset the rest themselves).
md.inversion.iscontrol = 0
md.masstransport.spcthickness = np.full(md.mesh.numberofvertices, np.nan)
md.smb.mass_balance = np.zeros(md.mesh.numberofvertices)

if save:
    print(f"Saving to {out_path}")
    pyissm.model.io.save_model(md, out_path)